# Paper figures

## break down of svgs

In [ ]:
from utils import svg2paths2, get_single_paths, disvg, drawing_to_tensor
import yaml
import random
from dataset import GlyphazznStage1Dataset
import matplotlib.pyplot as plt

with open('/home/mfeuerpfeil/master/thesis/configs/SVG_VQVAE_Stage1_figr8.yaml', 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

train_ds = GlyphazznStage1Dataset(**config["data_params"], train=False)


# fig.savefig(f"images/paper/figr8_{index}_original.png", bbox_inches='tight', pad_inches=0)


In [ ]:
# for index in [random.randint(0, len(train_ds)) for i in range(25)]:
import matplotlib
from svgpathtools import Path

index = 66

cmap = plt.get_cmap("tab20c")
colors_hex = [matplotlib.colors.rgb2hex(cmap(i)) for i in range(cmap.N)]

svg_path = train_ds.df.iloc[index].simplified_svg_file_path
paths, attributes, svg_attributes = svg2paths2(svg_path)

# single_paths = get_single_paths(paths)
single_paths = paths
single_paths = train_ds.get_similar_length_paths(single_paths, max_length=2.0)

new_single_paths = []
for i, path in enumerate(single_paths):
    # if i % 2 == 0:
    #     # print(path)
    #     # new_single_paths.append(Path(*train_ds.get_similar_length_paths([path], max_length=0.5)[1:]))
    #     new_single_paths.append(Path(*path._segments))
    # else:
    new_paths = train_ds.get_similar_length_paths([path], max_length=0.2)[1:]
    all_segs = []
    for myp in new_paths:
        all_segs.extend(myp._segments)
    new_single_paths.append(Path(*all_segs))
# print(new_single_paths)
# single_paths = get_single_paths(paths)
new_single_paths = [path for path in new_single_paths if path.length() > 0.]
# new_single_paths = [new_single_paths[i] for i in range(len(new_single_paths)) if not i%2 == 0]


# make alternating colors red, green blue for single paths
offset = 114
hex_colors = colors_hex[:6]
colors = ["black"] * offset + hex_colors + ["black"] * (len(new_single_paths) - len(hex_colors) - offset)
print(colors)

drawing = disvg(new_single_paths, colors=colors,paths2Drawing=True, stroke_widths=[0.7]*len(new_single_paths), viewbox = svg_attributes["viewBox"],dimensions=(1280,1280))

save = True

fig, ax = plt.subplots(1, 1, figsize=(10,10))
ax.imshow(drawing_to_tensor(drawing).permute(1,2,0))
ax.axis("off")
if save:
    fig.savefig(f"images/paper/figr8_{index}_split_colored.png", bbox_inches='tight', pad_inches=0)


fig, ax = plt.subplots(1, 1, figsize=(10,10))
ax.imshow(train_ds._get_full_svg_drawing(index, width=1280, as_tensor=True).permute(1,2,0))
ax.axis("off")
if save:
    fig.savefig(f"images/paper/figr8_{index}_original.png", bbox_inches='tight', pad_inches=0)

In [ ]:
len(new_single_paths[offset:offset+len(hex_colors)])

In [ ]:
len(new_single_paths[offset:offset+len(hex_colors)]), len(hex_colors)

In [ ]:
from utils import get_viewbox, raster
from torchvision.utils import make_grid
import torch
from typing import List

def get_rasterized_segments2(single_paths:list, stroke_width:float, total_max_diff: float, svg_attributes, centered = False, height: int = 128, width: int = 128, colors=None) -> List:
    if centered:
        single_paths = [my_path for my_path in single_paths if my_path.length() > 0.]
        if len(single_paths) == 0:
            # print("[INFO] tried to rasterize an empty path")
            return [torch.ones((3, height, width)), torch.ones((3, height, width))], [[width/2,height/2], [width/2,height/2]]
        out = [get_viewbox(my_path, total_max_diff) for my_path in single_paths]
        viewboxes = [x[0] for x in out]
        centers = [x[1] for x in out]
        if colors is not None:
            rasterized_segments = [raster(disvg(my_path, paths2Drawing=True, colors=[colors[i]], stroke_widths=[stroke_width] * len(my_path), viewbox=viewboxes[i]), out_h = height, out_w = width) for i, my_path in enumerate(single_paths)]
        else:
            rasterized_segments = [raster(disvg(my_path, paths2Drawing=True, stroke_widths=[stroke_width] * len(my_path), viewbox=viewboxes[i]), out_h = height, out_w = width) for i, my_path in enumerate(single_paths)]
        return rasterized_segments, centers
    else:
        viewbox=svg_attributes["viewBox"]
        rasterized_segments = [raster(disvg(my_path, paths2Drawing=True, stroke_widths=[stroke_width] * len(my_path), viewbox=viewbox), out_h = height, out_w = width) for my_path in single_paths if my_path.length() > 0.]
        centers = [(0,0)] * len(rasterized_segments)
        return rasterized_segments, centers

rasterized_segments, centers = get_rasterized_segments2(new_single_paths[offset:offset+len(hex_colors)], 0.4, 0.5, svg_attributes, centered=True, height=128, width=128, colors=hex_colors)
fig, ax = plt.subplots(1, 1, figsize=(10,10))
ax.imshow(make_grid(rasterized_segments, nrow = 3).permute(1,2,0))
ax.axis("off")
if save:
    fig.savefig(f"images/paper/figr8_{index}_colored_center_shapes.png", bbox_inches='tight', pad_inches=0)


## data preparation

In [ ]:
from dataset import GlyphazznStage1Dataset
import torch
import matplotlib.pyplot as plt
import numpy as np
import string
from utils import svg_to_tensor
from torchvision.utils import make_grid
import os
import random
from svgpathtools import svg2paths, svg2paths2, disvg, Path  # this is used to READ and breakdown SVG
from utils import get_similar_length_paths

In [ ]:
svg_path = "/scratch2/moritz_data/google_fonts/svgs_simplified/train/B/B_AbhayaLibre-Medium.svg"

paths, attributes, svg_attributes = svg2paths2(svg_path)
sim_length = get_similar_length_paths(paths, max_length = 15.)

In [ ]:
new_attributes = []
for i, attr in enumerate(attributes):
    new_attributes.append({
        "d" : attr["d"],
        "fill" : "none",
        "stroke" : ["red", "black", "blue"][i],
        "stroke-width" : "1",
    })

In [ ]:
# disvg(paths, svg_attributes = svg_attributes, paths2Drawing=True, stroke_widths=[0.9]*len(paths))
disvg(paths, svg_attributes = svg_attributes, paths2Drawing=True, attributes=new_attributes)

# Visualize GlyphazznStage1Dataset

In [ ]:
from dataset import GlyphazznStage1Dataset
import torch
import matplotlib.pyplot as plt
import numpy as np
import string
from utils import svg_to_tensor, svg2paths2, disvg
from torchvision.utils import make_grid
min_len = 1.

ds_8 = GlyphazznStage1Dataset(csv_path="/scratch2/moritz_data/figr8/stage1.csv",
                            channels=3,
                            width=128,
                            train = True,
                          individual_min_length=min_len,
                            individual_max_length=8.,
                            max_shapes_per_svg=1000,
                            stroke_width=1.06)

ds_5 = GlyphazznStage1Dataset(csv_path="/scratch2/moritz_data/figr8/stage1.csv",
                            channels=3,
                            width=128,
                            train = True,
                            individual_min_length=min_len,
                            individual_max_length=5.,
                            max_shapes_per_svg=1000,
stroke_width = 0.666)

ds_3 = GlyphazznStage1Dataset(csv_path="/scratch2/moritz_data/figr8/stage1.csv",
                            channels=3,
                            width=128,
                            train = True,
                            individual_min_length=min_len,
                            individual_max_length=3.,
                            max_shapes_per_svg=1000,
                            stroke_width = 0.4)

ds_15 = GlyphazznStage1Dataset(csv_path="/scratch2/moritz_data/figr8/stage1.csv",
                            channels=3,
                            width=128,
                            train = True,
                            individual_min_length=min_len,
                            individual_max_length=15.,
                            max_shapes_per_svg=1000,
                            stroke_width = 1.2)

## compare patch sizes of the same stroke

In [ ]:
import pandas as pd
df = pd.read_csv("/scratch2/moritz_data/figr8/stage1.csv")

In [ ]:
from svgpathtools import Path
import math
def crop_path_into_segments(path:Path, length:float = 5.):
    """
    a single input path is cropped into segments of approx length `length`. I say "approx" because we divide the path into same length segments, which will not be exactly `length` long.
    """
    segments = []
    try:
        num_iters = math.ceil(path.length() / length)
        for i in range(num_iters):
            cropped_segment = path.cropped(i/num_iters, (i+1)/num_iters)
            segments.append(cropped_segment)
    except Exception as e:
        print(e)
        pass
    return segments

def get_similar_length_paths(single_paths, max_length: float = 5.):
    """
    splits all the paths into similar length segments if they're too long
    """
    similar_length_paths = []
    for path in single_paths:
        try:
            segments = crop_path_into_segments(path, length=max_length)
            similar_length_paths.extend(segments)
        except AssertionError:
            print("Error while cropping path into segments, skipping...")
            continue
    return similar_length_paths

from utils import get_rasterized_segments

def path_to_stroke(simplified_svg_file_path, stroke_width, individual_max_length, width):

    svg_path = simplified_svg_file_path
    paths, attributes, svg_attributes = svg2paths2(svg_path)
    
    paths = [paths[0]]
    single_paths = get_similar_length_paths(paths, individual_max_length)
    
    single_paths = [path for path in single_paths if path.length() > 0.]

    rasterized_segments, centers = get_rasterized_segments(single_paths, stroke_width, individual_max_length, svg_attributes, centered=True, height=width, width=width)
    imgs = torch.stack(rasterized_segments)  # (n_shapes, channels, width, width)

    return imgs

idx = 29

length_3_img = path_to_stroke(df.iloc[idx].simplified_svg_file_path, 0.4, 3., 128)
length_3_img = make_grid(length_3_img, nrow=math.ceil(math.sqrt(len(length_3_img))))
length_5_img = path_to_stroke(df.iloc[idx].simplified_svg_file_path, 0.666, 5., 128)
length_5_img = make_grid(length_5_img, nrow=math.ceil(math.sqrt(len(length_5_img))))
length_8_img = path_to_stroke(df.iloc[idx].simplified_svg_file_path, 1.06, 8., 128)
length_8_img = make_grid(length_8_img, nrow=math.ceil(math.sqrt(len(length_8_img))))
length_15_img = path_to_stroke(df.iloc[idx].simplified_svg_file_path, 1.2, 15., 128)
length_15_img = make_grid(length_15_img, nrow=math.ceil(math.sqrt(len(length_15_img))))
length_full_img = path_to_stroke(df.iloc[idx].simplified_svg_file_path, 1.5, 100., 128)
length_full_img = make_grid(length_full_img, nrow=math.ceil(math.sqrt(len(length_15_img))))

In [ ]:
plt.imshow(length_full_img.permute(1,2,0))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10,10))
axes[1,1].imshow(length_3_img.permute(1,2,0))
axes[1,1].axis("off")
axes[1,1].set_title(f"Maximum patch size {round(3.0 / 72 * 100)}%")
axes[1,0].imshow(length_5_img.permute(1,2,0))
axes[1,0].axis("off")
axes[1,0].set_title(f"Maximum patch size  {round(5.0 / 72 * 100)}%")
axes[0,1].imshow(length_8_img.permute(1,2,0))
axes[0,1].axis("off")
axes[0,1].set_title(f"Maximum patch size  {round(8.0 / 72 * 100)}%")
axes[0,0].imshow(length_15_img.permute(1,2,0))
axes[0,0].axis("off")
axes[0,0].set_title(f"Maximum patch size  {round(15.0 / 72 * 100)}%")
fig.suptitle("Different patch sizes for the same stroke")


In [ ]:
from torchvision.utils import save_image
for img, length in zip([length_3_img, length_5_img, length_8_img, length_15_img, length_full_img], [3, 5, 8, 15, 89]):
    save_image(img, f"images/stage1data/stroke_4_{min(round(length / 72 * 100), 100)}_percent_patch_size.png")

---

In [ ]:
import random
fig, ax = plt.subplots(1, 1, figsize=(14,20))
rand_indx = random.randint(0, len(ds_8))
ax.imshow(make_grid(ds_15[2][0],nrow=20).permute(1,2,0))

In [ ]:
import time
from tqdm import tqdm
import random
import json

def get_fraction_dict(all_num_shapes_7:np.ndarray):
    return_dict = {
        "len" : len(all_num_shapes_7),
        "frac_54" : (all_num_shapes_7 <= 54).sum() / len(all_num_shapes_7),
        "frac_64" : (all_num_shapes_7 <= 64).sum() / len(all_num_shapes_7),
        "frac_118" : (all_num_shapes_7 <= 118).sum() / len(all_num_shapes_7),
        "frac_128" : (all_num_shapes_7 <= 128).sum() / len(all_num_shapes_7),
        "frac_246" : (all_num_shapes_7 <= 246).sum() / len(all_num_shapes_7),
        "frac_256" : (all_num_shapes_7 <= 256).sum() / len(all_num_shapes_7),
        "frac_502" : (all_num_shapes_7 <= 502).sum() / len(all_num_shapes_7),
        "frac_512" : (all_num_shapes_7 <= 512).sum() / len(all_num_shapes_7),
    }
    print(return_dict)
    return return_dict
all_dicts = []
all_num_shapes_7 = []
all_num_shapes_5 = []
all_num_shapes_3 = []

random_idxs = random.sample(range(len(ds_7)), 50000)

for i in tqdm(random_idxs, total=len(random_idxs)):
    all_num_shapes_7.append(ds_7[i][0].shape[0])

for i in tqdm(random_idxs, total=len(random_idxs)):
    all_num_shapes_5.append(ds_5[i][0].shape[0])

for i in tqdm(random_idxs, total=len(random_idxs)):
    all_num_shapes_3.append(ds_3[i][0].shape[0])

all_dicts.append(get_fraction_dict(np.array(all_num_shapes_7)))
all_dicts.append(get_fraction_dict(np.array(all_num_shapes_5)))
all_dicts.append(get_fraction_dict(np.array(all_num_shapes_3)))
with open("/scratch2/moritz_data/figr8/stats/fraction_dicts.json", "w") as f:
    json.dump(all_dicts, f)
np.save("/scratch2/moritz_data/figr8/stats/all_num_shapes_7.npy", np.array(all_num_shapes_7))
np.save("/scratch2/moritz_data/figr8/stats/all_num_shapes_5.npy", np.array(all_num_shapes_5))
np.save("/scratch2/moritz_data/figr8/stats/all_num_shapes_3.npy", np.array(all_num_shapes_3))

print("DONE")



In [ ]:
import json
with open("fraction_dicts_7.json", "w") as f:
    json.dump(all_dicts, f)

In [ ]:
all_num_shapes_7 = np.array([0, 54, 212, 233.2, 11])

(all_num_shapes_7 > 1).sum() / len(all_num_shapes_7)

# np.mean(all_num_shapes_7)

In [ ]:
def show(sample):
    grid_image = make_grid(sample, nrow=8, padding=2, normalize=True)

    # Plot the grid image
    plt.imshow(grid_image.permute(1, 2, 0))
    plt.axis('off')
    plt.show()

In [ ]:
ds = ds_3
max_len=3.
random_indices = np.random.randint(0, len(ds), size=5)

fig, axs = plt.subplots(5, 3, figsize=(10, 15))
fig.suptitle(f"{min_len} <= stroke length <= {max_len}")

for i, random_idx in enumerate(random_indices):
    example = ds[random_idx]
    full_svg = ds._get_full_svg_drawing(random_idx, as_tensor=True)
    class_char = string.printable[example[1][0].int().item()]

    mean_reduction = torch.mean(example[0], dim=0)
    min_reduction = torch.min(example[0], dim=0)[0]

    axs[i, 0].imshow(full_svg.permute(1, 2, 0))
    axs[i, 0].set_title(f"full svg: {class_char}")
    axs[i, 0].set_xticks([])
    axs[i, 0].set_yticks([])
    
    axs[i, 1].imshow(mean_reduction.permute(1, 2, 0))
    axs[i, 1].set_title("mean reduction")
    axs[i, 1].set_xticks([])
    axs[i, 1].set_yticks([])
    
    axs[i, 2].imshow(min_reduction.permute(1, 2, 0))
    axs[i, 2].set_title("min reduction")
    axs[i, 2].set_xticks([])
    axs[i, 2].set_yticks([])

# plt.savefig(f"images/stage1data/data_sample_{random_indices[0] + random_indices[1]}.png")
plt.show()

In [ ]:
example = ds[random_indices[1]]
show(example[0])

# Visualize Stage 2 dataset

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
import os
import yaml
from dataset import VQDataModule
from models import VQ_SVG_Stage2, Vector_VQVAE
from tokenizer import VQTokenizer
from experiment import SVG_VQVAE_Stage2_Experiment
import torch
import random
import matplotlib.pyplot as plt
import numpy as np
import torchvision.utils as vutils
from PIL import Image
from torch import Tensor
import string

import pandas as pd
device = "cuda" if torch.cuda.is_available() else "cpu"
from utils import calculate_global_positions, shapes_to_drawing, drawing_to_tensor
from svg_fixing import get_fixed_svg_drawing, get_fixed_svg_render

## dataloader

In [ ]:
CONFIG_PATH = "configs/SVG_VQVAE_Stage2.yaml"
device = "cuda" if torch.cuda.is_available() else "cpu"

with open(CONFIG_PATH, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

config["data_params"]["csv_path"] = "/scratch2/moritz_data/glyphazzn/tokenized/split.csv"
data = VQDataModule(**config["data_params"], context_length=config['model_params']['max_seq_len'])
data.setup()

train_ds = data.train_dataset

vq_model = Vector_VQVAE(**config['stage1_params'], device = device)
state_dict = torch.load(config['stage1_params']["checkpoint_path"])["state_dict"]
try:
    vq_model.load_state_dict(state_dict)
except:
    vq_model.load_state_dict({k.replace("model.", ""): v for k, v in state_dict.items()})
vq_model = vq_model.eval()
tokenizer = VQTokenizer(vq_model, config["data_params"]["width"], 1, "bert-base-uncased", device = device)

df = pd.read_csv("/scratch2/moritz_data/glyphazzn/svgs_simplified/split.csv")
df.head()

df_train = df[df["split"] == "train"].reset_index(drop=True)
df_test = df[df["split"] == "test"].reset_index(drop=True)

In [ ]:
text_tokens, att_mask, vq_tokens, targets, pad_tok = train_ds[:25]
vq_tokens = vq_tokens.to(device)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import torch
from torchvision.utils import make_grid

def render_font_characters_to_images(font_path, size = 128):
    # Define the range of ASCII printable characters
    chars = string.printable[:62]
    character_images = []
    characters = []
    
    # Load the font
    try:
        font = ImageFont.truetype(font_path, size=int(size*0.7))  # You can adjust the size as needed
    except IOError:
        raise ValueError("Font cannot be loaded. Please check the path and the font file.")
    
    for i, char in enumerate(chars):
        try:
            # Create a blank image with a white background
            image = Image.new('RGB', (size, size), 'white')  # Adjust size as needed
            draw = ImageDraw.Draw(image)
            
            # Draw the character
            draw.text((20, 20), char, font=font, stroke_fill="black", stroke_width = 1)  # Adjust position as needed
            
            # Convert the PIL image to a PyTorch tensor
            tensor = torch.tensor(np.array(image)).permute(2, 0, 1)  # Convert to channels first format
            character_images.append(tensor)
            characters.append(char)
        except Exception as e:
            print(f"Could not render character '{char}': {e}")

    return character_images, characters

idx = random.randint(0, len(df_train))

# Example usage
font_path = df_train.iloc[idx].font_path
font = df_train.iloc[idx].font
images, chars = render_font_characters_to_images(font_path)
print(f"Rendered {len(images)} characters.")

fig, ax = plt.subplots(1, 1, figsize=(8,8))
ax.imshow(make_grid(images, nrow=9, padding=2).permute(1,2,0))
ax.axis('off')
ax.set_title(font+", 0-9, a-z, A-Z")
plt.show()

In [ ]:
font

In [ ]:
df_train.iloc[5].font_path

In [ ]:

fig, axes = plt.subplots(5, 5, figsize=(20, 20))
for i, ax in enumerate(axes.flatten()):
    bezier_points, positions = tokenizer.decode(vq_tokens[i], ignore_special_tokens=True)
    ax.imshow(get_fixed_svg_render(bezier_points, positions, method="min_dist_clip").permute(1, 2, 0))
    ax.set_title(f"idx {i} "+tokenizer.decode_text(text_tokens[i].to(device)))
    ax.axis('off')